# 02 — Matched Ablation, Router Sensitivity, and Component-wise YoY Analysis

This notebook is intentionally separated from the primary benchmark. Its purpose is to test **mechanistic claims** without changing the frozen headline protocol.

Three analyses are performed:

1. structural and feature ablations under matched hyperparameter optimization;
2. router hyperparameter sensitivity, temporal calibration stability, and nonnegative least-squares alternatives;
3. separate introduction of annual-information components on the two long primary datasets.

The external one-year BDG series are not used for year-over-year component analysis because they do not provide a meaningful prior-year history across the evaluation window.

## 1. Setup

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lash_revision_core import ExperimentConfig
from lash_revision_ablation import (
    STRUCTURAL_VARIANTS, FEATURE_VARIANTS, YOY_VARIANTS,
    run_matched_ablation_suite, run_router_sensitivity_suite,
    run_yoy_component_suite,
)

RUN_PROFILE = "paper"
config = ExperimentConfig(
    data_root=DATA_DIR,
    output_root=OUTPUT_DIR,
    run_profile=RUN_PROFILE,
    weather_mode="historical_only",
    hpo_seeds=(42, 142, 242),
    final_refit_seeds=(42, 142, 242),
    primary_hpo_repeats=3,
    external_hpo_repeats=3,
    save_models=True,
)

print("Structural variants:", list(STRUCTURAL_VARIANTS))
print("Feature variants:", list(FEATURE_VARIANTS))
print("YoY variants:", list(YOY_VARIANTS))

## 2. Router sensitivity using frozen expert predictions

This stage does **not retrain** the experts. It reuses the calibration predictions saved by Notebook 01 and examines:

- weight-grid steps: `0.01, 0.05, 0.10`;
- smoothing windows: `none, 3, 5`;
- shrinkage toward the global weight: `0, 0.5, 1`;
- hybrid acceptance thresholds: `0, 0.0025, 0.005, 0.01` percentage points;
- three chronological calibration blocks for temporal stability;
- global and horizon-specific nonnegative least-squares (NNLS) alternatives.

All router choices are determined on the validation-calibration segment and are then applied unchanged to frozen test expert predictions.

In [ ]:
router_results = run_router_sensitivity_suite(config)
for key, df in router_results.items():
    print(key)
    display(df.sort_values("selection_score").head(15))

## 3. Structural and feature ablations with matched optimization

Every ablated sequential architecture is retuned from scratch with the same model-specific HPO budget as the full LASH sequential expert. Feature-changing variants rebuild both the sequential and Ridge designs, retune them, recalibrate the router, and refit on train+validation before test evaluation.

For interpretability, each variant reports two outputs separately:

- `SEQUENTIAL_ONLY` — isolates the sequential expert itself;
- `ROUTED_PIPELINE` — reports the behavior of the complete selective hybrid after calibration.

This separation prevents a Ridge-only router decision from being misinterpreted as direct evidence about a removed neural component.

In [ ]:
ablation_results = run_matched_ablation_suite(
    config,
    dataset_keys=("CLUSTER_1", "CLUSTER_2"),
)
for key, result in ablation_results.items():
    print(key)
    display(result["summary"].sort_values(["component", "selection_score_mean"]))

## 4. Component-wise annual-information sensitivity

The no-year-over-year design is evaluated against separate annual components rather than one bundled reverse ablation:

- prior-year demand lag only;
- holiday-matched prior-year proxy only;
- annual scaling only;
- annual anchor component only;
- all annual components together.

Every configuration is selected using the same chronological validation protocol. The resulting evidence should be interpreted as dataset-specific sensitivity, not as a universal claim that annual information is always beneficial or harmful.

In [ ]:
yoy_results = run_yoy_component_suite(
    config,
    dataset_keys=("CLUSTER_1", "CLUSTER_2"),
)
for key, result in yoy_results.items():
    print(key)
    display(result["summary"].sort_values(["component", "selection_score_mean"]))

## 5. Output locations

The notebook writes separate Excel workbooks under:

- `outputs/router_sensitivity/<dataset>/`
- `outputs/ablation/<dataset>/`
- `outputs/yoy/<dataset>/`

The separation keeps the primary benchmark immutable and makes it possible to release sensitivity scripts and result tables without mixing them with headline model selection.